<a href="https://colab.research.google.com/github/weloo11/mnist-image-classifier/blob/main/finalsamehinshallah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
from tensorflow.keras.datasets import mnist
from sklearn.decomposition import PCA
from skimage.feature import hog


# ==========================================================
# 1) FEATURE EXTRACTION
# ==========================================================

def flatten_features(images):
    return images.reshape(images.shape[0], -1)


def hog_single_image(image):
    return hog(
        image,
        orientations=9,
        pixels_per_cell=(4, 4),   # improved HOG
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )


def hog_features_dataset(images):
    features = []

    for i in range(images.shape[0]):
        features.append(hog_single_image(images[i]))

    return np.array(features, dtype=np.float64)


# ==========================================================
# 2) STRATIFIED TRAIN / VALIDATION SPLIT
# ==========================================================

def stratified_train_val_split(x_train, y_train, val_ratio=0.15, random_state=42):
    np.random.seed(random_state)

    class_0_idx = np.where(y_train == 0)[0]
    class_1_idx = np.where(y_train == 1)[0]

    np.random.shuffle(class_0_idx)
    np.random.shuffle(class_1_idx)

    val_0_size = int(len(class_0_idx) * val_ratio)
    val_1_size = int(len(class_1_idx) * val_ratio)

    val_idx = np.concatenate([
        class_0_idx[:val_0_size],
        class_1_idx[:val_1_size]
    ])

    train_idx = np.concatenate([
        class_0_idx[val_0_size:],
        class_1_idx[val_1_size:]
    ])

    np.random.shuffle(train_idx)
    np.random.shuffle(val_idx)

    return (
        x_train[train_idx],
        y_train[train_idx],
        x_train[val_idx],
        y_train[val_idx]
    )


# ==========================================================
# 3) PREPROCESSING
# ==========================================================

def preprocess_mnist(
    target_digit=5,
    method="hog",
    pca_components=100,
    val_ratio=0.15
):
    (x_train, y_train), (x_test, y_test) = mnist.load_data()

    y_train = np.where(y_train == target_digit, 1, 0)
    y_test = np.where(y_test == target_digit, 1, 0)

    x_train = x_train.astype(np.float64) / 255.0
    x_test = x_test.astype(np.float64) / 255.0

    x_train, y_train, x_val, y_val = stratified_train_val_split(
        x_train,
        y_train,
        val_ratio=val_ratio,
        random_state=42
    )

    if method == "flatten":
        print("Using Flatten features")

        X_train = flatten_features(x_train)
        X_val = flatten_features(x_val)
        X_test = flatten_features(x_test)

    elif method == "pca":
        print("Using PCA features")

        X_train_flat = flatten_features(x_train)
        X_val_flat = flatten_features(x_val)
        X_test_flat = flatten_features(x_test)

        pca = PCA(n_components=pca_components)

        X_train = pca.fit_transform(X_train_flat)
        X_val = pca.transform(X_val_flat)
        X_test = pca.transform(X_test_flat)

        print("PCA variance kept:", round(np.sum(pca.explained_variance_ratio_), 4))

    elif method == "hog":
        print("Using improved HOG features")

        X_train = hog_features_dataset(x_train)
        X_val = hog_features_dataset(x_val)
        X_test = hog_features_dataset(x_test)

    else:
        raise ValueError("method must be 'flatten', 'pca', or 'hog'")

    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)

    std[std == 0] = 1.0

    X_train = (X_train - mean) / std
    X_val = (X_val - mean) / std
    X_test = (X_test - mean) / std

    print("X_train:", X_train.shape)
    print("X_val:", X_val.shape)
    print("X_test:", X_test.shape)

    print("Train positives:", np.sum(y_train == 1))
    print("Train negatives:", np.sum(y_train == 0))
    print("Val positives:", np.sum(y_val == 1))
    print("Val negatives:", np.sum(y_val == 0))
    print("Test positives:", np.sum(y_test == 1))
    print("Test negatives:", np.sum(y_test == 0))

    return X_train, y_train, X_val, y_val, X_test, y_test, mean, std


# ==========================================================
# 4) METRICS
# ==========================================================

def confusion_matrix_manual(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    return np.array([[tn, fp],
                     [fn, tp]])


def accuracy_score_manual(y_true, y_pred):
    return np.mean(y_true == y_pred)


def precision_score_manual(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    if tp + fp == 0:
        return 0.0

    return tp / (tp + fp)


def recall_score_manual(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    if tp + fn == 0:
        return 0.0

    return tp / (tp + fn)


def f1_score_manual(y_true, y_pred):
    precision = precision_score_manual(y_true, y_pred)
    recall = recall_score_manual(y_true, y_pred)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


# ==========================================================
# 5) LINEAR SVM FROM SCRATCH
# ==========================================================

class LinearSVMFromScratch:
    def __init__(
        self,
        learning_rate=0.01,
        lambda_param=0.0005,
        n_epochs=60,
        batch_size=512,
        lr_decay=0.95,
        patience=8
    ):
        self.learning_rate = learning_rate
        self.lambda_param = lambda_param
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.lr_decay = lr_decay
        self.patience = patience

        self.w = None
        self.b = 0.0

        self.train_losses = []
        self.best_w = None
        self.best_b = None
        self.best_f1 = -1.0

    def _compute_class_weights(self, y):
        n_samples = len(y)
        n_pos = np.sum(y == 1)
        n_neg = np.sum(y == -1)

        weight_pos = n_samples / (2 * n_pos)
        weight_neg = n_samples / (2 * n_neg)

        return weight_pos, weight_neg

    def _hinge_loss(self, X, y, weight_pos, weight_neg):
        scores = X @ self.w + self.b
        margins = y * scores

        sample_weights = np.where(y == 1, weight_pos, weight_neg)
        hinge = np.maximum(0, 1 - margins)

        loss = self.lambda_param * np.dot(self.w, self.w) + np.mean(sample_weights * hinge)

        return loss

    def fit(self, X, y, X_val=None, y_val=None):
        n_samples, n_features = X.shape

        self.w = np.zeros(n_features, dtype=np.float64)
        self.b = 0.0

        weight_pos, weight_neg = self._compute_class_weights(y)

        current_lr = self.learning_rate
        no_improve_count = 0

        for epoch in range(self.n_epochs):
            indices = np.random.permutation(n_samples)

            X_shuffled = X[indices]
            y_shuffled = y[indices]

            for start in range(0, n_samples, self.batch_size):
                end = start + self.batch_size

                X_batch = X_shuffled[start:end]
                y_batch = y_shuffled[start:end]

                scores = X_batch @ self.w + self.b
                margins = y_batch * scores

                sample_weights = np.where(y_batch == 1, weight_pos, weight_neg)
                active = margins < 1

                if np.any(active):
                    X_active = X_batch[active]
                    y_active = y_batch[active]
                    w_active = sample_weights[active]

                    grad_w_hinge = -np.mean(
                        (w_active * y_active)[:, np.newaxis] * X_active,
                        axis=0
                    )

                    grad_b_hinge = -np.mean(w_active * y_active)

                else:
                    grad_w_hinge = np.zeros_like(self.w)
                    grad_b_hinge = 0.0

                grad_w = 2 * self.lambda_param * self.w + grad_w_hinge
                grad_b = grad_b_hinge

                self.w -= current_lr * grad_w
                self.b -= current_lr * grad_b

            train_loss = self._hinge_loss(X, y, weight_pos, weight_neg)
            self.train_losses.append(train_loss)

            msg = f"Epoch {epoch + 1}/{self.n_epochs}, Loss: {train_loss:.6f}"

            if X_val is not None and y_val is not None:
                y_val_pred = self.predict(X_val)

                y_val_binary = np.where(y_val == 1, 1, 0)
                y_val_pred_binary = np.where(y_val_pred == 1, 1, 0)

                val_f1 = f1_score_manual(y_val_binary, y_val_pred_binary)

                msg += f", Val F1: {val_f1:.6f}"

                if val_f1 > self.best_f1:
                    self.best_f1 = val_f1
                    self.best_w = self.w.copy()
                    self.best_b = self.b
                    no_improve_count = 0
                else:
                    no_improve_count += 1

                if no_improve_count >= self.patience:
                    print(msg)
                    print("Early stopping triggered.")
                    break

            print(msg)
            current_lr *= self.lr_decay

        if self.best_w is not None:
            self.w = self.best_w
            self.b = self.best_b

    def decision_function(self, X):
        return X @ self.w + self.b

    def predict(self, X):
        scores = self.decision_function(X)
        return np.where(scores >= 0, 1, -1)


# ==========================================================
# 6) RUN
# ==========================================================

method = "hog"       # change here: "flatten", "pca", or "hog"
target_digit = 5

X_train, y_train, X_val, y_val, X_test, y_test, mean, std = preprocess_mnist(
    target_digit=target_digit,
    method=method,
    pca_components=100,
    val_ratio=0.15
)

y_train_svm = np.where(y_train == 1, 1, -1)
y_val_svm = np.where(y_val == 1, 1, -1)
y_test_svm = np.where(y_test == 1, 1, -1)

svm_model = LinearSVMFromScratch(
    learning_rate=0.01,
    lambda_param=0.0005,
    n_epochs=60,
    batch_size=512,
    lr_decay=0.95,
    patience=8
)

svm_model.fit(
    X_train,
    y_train_svm,
    X_val=X_val,
    y_val=y_val_svm
)


# ==========================================================
# 7) EVALUATION
# ==========================================================

y_val_pred_svm = svm_model.predict(X_val)
y_val_pred = np.where(y_val_pred_svm == 1, 1, 0)

print("\n===== VALIDATION RESULTS =====")
print("Accuracy :", accuracy_score_manual(y_val, y_val_pred))
print("Precision:", precision_score_manual(y_val, y_val_pred))
print("Recall   :", recall_score_manual(y_val, y_val_pred))
print("F1-score :", f1_score_manual(y_val, y_val_pred))
print("Confusion Matrix:\n", confusion_matrix_manual(y_val, y_val_pred))

y_test_pred_svm = svm_model.predict(X_test)
y_test_pred = np.where(y_test_pred_svm == 1, 1, 0)

print("\n===== TEST RESULTS =====")
print("Accuracy :", accuracy_score_manual(y_test, y_test_pred))
print("Precision:", precision_score_manual(y_test, y_test_pred))
print("Recall   :", recall_score_manual(y_test, y_test_pred))
print("F1-score :", f1_score_manual(y_test, y_test_pred))
print("Confusion Matrix:\n", confusion_matrix_manual(y_test, y_test_pred))

Using improved HOG features
X_train: (51001, 1296)
X_val: (8999, 1296)
X_test: (10000, 1296)
Train positives: 4608
Train negatives: 46393
Val positives: 813
Val negatives: 8186
Test positives: 892
Test negatives: 9108
Epoch 1/60, Loss: 0.152264, Val F1: 0.722321
Epoch 2/60, Loss: 0.078651, Val F1: 0.871405
Epoch 3/60, Loss: 0.058491, Val F1: 0.906109
Epoch 4/60, Loss: 0.046226, Val F1: 0.904681
Epoch 5/60, Loss: 0.044635, Val F1: 0.893830
Epoch 6/60, Loss: 0.038742, Val F1: 0.922900
Epoch 7/60, Loss: 0.032562, Val F1: 0.922722
Epoch 8/60, Loss: 0.029990, Val F1: 0.923875
Epoch 9/60, Loss: 0.028147, Val F1: 0.935522
Epoch 10/60, Loss: 0.028075, Val F1: 0.927620
Epoch 11/60, Loss: 0.029470, Val F1: 0.918950
Epoch 12/60, Loss: 0.024815, Val F1: 0.931315
Epoch 13/60, Loss: 0.024710, Val F1: 0.938489
Epoch 14/60, Loss: 0.027084, Val F1: 0.911149
Epoch 15/60, Loss: 0.025110, Val F1: 0.935050
Epoch 16/60, Loss: 0.026632, Val F1: 0.916905
Epoch 17/60, Loss: 0.021303, Val F1: 0.933721
Epoch 18/